In [1]:
!pip install -q \
groq \
yfinance \
transformers \
torch \
pandas \
numpy \
scikit-learn \
requests \
tqdm \
rich \
ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 87.9 MB/s eta 0:00:00


In [2]:
import os
import sys
import json
import time
import warnings
import textwrap
import logging
from datetime import datetime, date
from collections import defaultdict

warnings.filterwarnings("ignore")

for _log in ("yfinance", "peewee", "urllib3", "requests", "transformers"):
    logging.getLogger(_log).setLevel(logging.CRITICAL)

import pandas as pd
import numpy as np
import yfinance as yf

# Groq client
from groq import Groq

from tqdm.auto import tqdm

# HuggingFace / FinBERT
import torch
from transformers import pipeline

# Display
from rich.console import Console
from rich.panel import Panel
from rich.rule import Rule


def _ok(msg):
    print(f"✅ {msg}", flush=True)


def _info(msg):
    print(f"ℹ️  {msg}", flush=True)


def _warn(msg):
    print(f"⚠️  {msg}", flush=True)


def _err(msg):
    print(f"❌ {msg}", flush=True)


def _hr(title=""):
    if title:
        print(f"\n{'─'*60}  {title}\n", flush=True)
    else:
        print(f"\n{'─'*60}\n", flush=True)

In [3]:
import torch
from transformers import pipeline


class SentimentModel:
    POSITIVE = {
        "beat","surge","jump","rise","gain","profit","strong","growth","record",
        "rally","upgrade","buy","outperform","exceed","higher","boost","revenue",
        "positive","bullish","breakthrough","innovation","expansion","dividend",
        "earnings","overweight","launch","partnership","win","recover","rebound",
        "soar","peak","milestone","deal","acquisition","approve","success",
        "increase","advance","upside","raised","guidance","momentum","optimistic",
    }

    NEGATIVE = {
        "miss","fall","drop","decline","loss","weak","cut","downgrade","sell",
        "underperform","lower","reduce","bearish","concern","risk","warn",
        "disappoint","crash","plunge","slump","layoff","lawsuit","recall",
        "investigation","penalty","fine","debt","default","bankruptcy","fraud",
        "dispute","delay","cancel","fail","negative","decrease","shrink","below",
        "poor","pressure","challenge","threat","halt","suspend","downside",
        "lowered","slowdown","pessimistic","cautious",
    }

    NEGATORS = {
        "not","no","never","neither","barely","hardly","scarcely","without"
    }

    def __init__(self, use_finbert=True):
        self.pipe = None
        self.use_finbert = False

        if use_finbert:
            self._load_finbert()

    def _load_finbert(self):
        try:
            print("Loading FinBERT… (downloads ~500 MB on first run)")

            device = 0 if torch.cuda.is_available() else -1

            self.pipe = pipeline(
                "sentiment-analysis",
                model="ProsusAI/finbert",
                tokenizer="ProsusAI/finbert",
                device=device,
                truncation=True,
                max_length=512,
            )

            self.use_finbert = True
            _ok(f"FinBERT loaded ({'GPU' if device == 0 else 'CPU'})")

        except Exception as e:
            self.pipe = None
            self.use_finbert = False
            _warn(f"FinBERT unavailable ({e}); using lexicon fallback")

    def _lexicon_score(self, text: str) -> float:
        words = text.lower().split()

        score = 0.0
        negated = False

        for w in words:
            token = "".join(ch for ch in w if ch.isalpha())

            if token in self.NEGATORS:
                negated = True
                continue

            if token in self.POSITIVE:
                score += -0.8 if negated else 1.0
                negated = False

            elif token in self.NEGATIVE:
                score += 0.8 if negated else -1.0
                negated = False

            else:
                negated = False

        score = score / max(1, len(words) * 0.12)
        return max(-1.0, min(1.0, score))

    def score_batch(self, texts, batch_size=32, pbar=None):

        if self.use_finbert and self.pipe:

            results = []

            for i in range(0, len(texts), batch_size):

                batch = texts[i:i + batch_size]

                try:
                    outputs = self.pipe(
                        batch,
                        batch_size=batch_size,
                        truncation=True,
                        max_length=512,
                    )

                    for r in outputs:
                        label = r["label"].lower()
                        score = r["score"]

                        if label == "positive":
                            results.append(score)

                        elif label == "negative":
                            results.append(-score)

                        else:
                            results.append(0.0)

                except Exception:
                    results.extend(self._lexicon_score(t) for t in batch)

                if pbar:
                    pbar.update(len(batch))

            return results

        # Lexicon fallback
        results = []

        for text in texts:
            results.append(self._lexicon_score(text))

            if pbar:
                pbar.update(1)

        return results

    def score(self, text: str) -> float:
        return self.score_batch([text])[0]

In [4]:
def load_dataset() -> pd.DataFrame:
    _info("Loading dataset: analyst_ratings_processed.csv")

    df = pd.read_csv(
        "analyst_ratings_processed.csv",
        engine="python",
        on_bad_lines="skip",
    )

    df.columns = [c.strip().lower() for c in df.columns]

    rename = {}
    for col in df.columns:
        if "title" in col or "headline" in col:
            rename[col] = "title"
        elif "date" in col or "time" in col:
            rename[col] = "date"
        elif "stock" in col or "ticker" in col or "symbol" in col:
            rename[col] = "stock"

    df = df.rename(columns=rename)

    required = {"title", "date", "stock"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"CSV missing columns: {missing}. Found: {list(df.columns)}"
        )

    df["stock"] = (
        df["stock"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    df = df.dropna(subset=["title", "stock"])

    df["date"] = pd.to_datetime(
        df["date"],
        utc=True,
        errors="coerce"
    )

    df = df.dropna(subset=["date"])

    _ok(
        f"Loaded {len(df):,} headlines · "
        f"{df['stock'].nunique():,} stocks "
        f"({df['date'].min().date()} → {df['date'].max().date()})"
    )

    return df


def pick_candidates(df: pd.DataFrame, n: int = 100, extra: int = 40) -> list:
    counts = df["stock"].value_counts()
    pool = counts.head(n + extra).index.tolist()

    _ok(
        f"Candidate pool: {len(pool)} tickers "
        f"(top {n} + {extra} spares for delistings)"
    )

    return pool


def filter_active(candidates: list, prices: dict, n: int = 100) -> list:
    active = [s for s in candidates if s in prices]
    dropped = [s for s in candidates if s not in prices]
    final = active[:n]

    _ok(
        f"Active: {len(active)} | "
        f"Delisted/no-data: {len(dropped)} → using top {len(final)}"
    )

    if dropped:
        print(f"Dropped: {', '.join(dropped)}")

    return final

In [5]:
def fetch_historical_prices(
    tickers: list,
    start: str,
    end: str,
    pause: float = 0.15,
) -> dict:

    prices = {}
    delisted = []
    errored = []

    print(f"\nFetching price history for {len(tickers)} tickers…")

    pbar = tqdm(
        tickers,
        desc="Price fetch",
        unit="ticker",
        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]",
    )

    for sym in pbar:

        pbar.set_postfix(ticker=sym, refresh=False)

        try:

            hist = yf.download(
                sym,
                start=start,
                end=end,
                auto_adjust=True,
                progress=False,
                threads=False,
            )

            if hist.empty:
                delisted.append(sym)
                continue

            hist.index = pd.to_datetime(hist.index).tz_localize(None)

            # -------- FIX FOR NEW YFINANCE --------
            if isinstance(hist.columns, pd.MultiIndex):
                hist.columns = hist.columns.get_level_values(0)
            # -------------------------------------

            prices[sym] = hist[
                ["Open", "High", "Low", "Close", "Volume"]
            ].copy()

        except Exception as e:
            errored.append((sym, str(e)[:80]))

        time.sleep(pause)

    pbar.close()

    print(
        f"\n✅ Price data: {len(prices)} loaded | "
        f"{len(delisted)} delisted/empty | "
        f"{len(errored)} errors"
    )

    if delisted:
        print("No data:", ", ".join(delisted))

    if errored:
        for sym, msg in errored:
            print(f"✗ {sym}: {msg}")

    return prices


def fetch_current_price(ticker: str):

    try:

        tk = yf.Ticker(ticker)

        info = tk.info

        hist = tk.history(
            period="5d",
            auto_adjust=True,
        )

        if hist.empty:
            raise ValueError("No price data")

        hist = hist.dropna()

        cur = float(hist["Close"].iloc[-1])

        prev = (
            float(hist["Close"].iloc[-2])
            if len(hist) >= 2
            else cur
        )

        return {
            "symbol": ticker,
            "current_price": cur,
            "prev_close": prev,
            "open": float(hist["Open"].iloc[-1]),
            "high": float(hist["High"].iloc[-1]),
            "low": float(hist["Low"].iloc[-1]),
            "volume": int(hist["Volume"].iloc[-1]),
            "day_change_pct": (cur - prev) / prev * 100,
            "name": info.get("longName", ticker),
            "sector": info.get("sector", "Unknown"),
        }

    except Exception as e:

        _warn(f"Price fetch failed for {ticker}: {e}")

        return {
            "symbol": ticker,
            "current_price": None,
            "error": str(e),
        }


def fetch_today_news(ticker: str):

    try:

        raw = yf.Ticker(ticker).news or []

        news = []

        for item in raw[:15]:

            title = (
                item.get("title")
                or item.get("content", {}).get("title", "")
            )

            if not title:
                continue

            news.append(
                {
                    "title": title,
                    "publisher": item.get("publisher", ""),
                    "link": item.get("link", ""),
                    "date": datetime.fromtimestamp(
                        item.get(
                            "providerPublishTime",
                            time.time(),
                        )
                    ).isoformat(),
                }
            )

        return news

    except Exception as e:

        _warn(f"News fetch failed for {ticker}: {e}")

        return []

In [6]:
N_BANDS = 20

def score_to_band(score):
    band = int(((score + 1) / 2) * N_BANDS)
    return max(0, min(N_BANDS - 1, band))


def build_band_tables(df, prices, model, top_stocks):

    results = {}

    stock_df = df[df.stock.isin(top_stocks)].copy()
    total_hl = len(stock_df)

    print(f"\nBuilding band tables for {len(top_stocks)} stocks ({total_hl:,} headlines)\n")

    stock_bar = tqdm(top_stocks, desc="Stocks")

    for sym in stock_bar:

        if sym not in prices:
            continue

        sub = stock_df[stock_df.stock == sym].copy()

        if len(sub) < 10:
            continue

        # ---------------- SENTIMENT ----------------
        scores = model.score_batch(sub["title"].tolist())
        sub["score"] = scores

        sub["date"] = pd.to_datetime(sub["date"], errors="coerce").dt.date
        sub = sub.dropna(subset=["date"])

        daily = (
            sub.groupby("date")["score"]
            .mean()
            .reset_index()
            .rename(columns={"score": "sentiment"})
        )

        # ---------------- PRICE (ROBUST FIX) ----------------
        price = prices[sym].copy().reset_index()

        # normalize column names
        price.columns = [str(c).lower() for c in price.columns]

        # detect date column safely
        date_col = price.columns[0]
        price["date"] = pd.to_datetime(price[date_col], errors="coerce").dt.date

        # ensure required column exists
        if "close" not in price.columns:
            continue

        price = price.dropna(subset=["date", "close"])

        # compute next-day return safely
        price["next_close"] = price["close"].shift(-1)
        price["next_return"] = (price["next_close"] - price["close"]) / price["close"] * 100

        price = price.dropna(subset=["next_return"])

        # ---------------- MERGE ----------------
        merged = pd.merge(
            daily,
            price[["date", "next_return"]],
            on="date",
            how="inner",
        )

        if len(merged) < 5:
            continue

        merged["band"] = merged["sentiment"].apply(score_to_band)

        band_returns = np.zeros(N_BANDS)
        band_counts = np.zeros(N_BANDS, dtype=int)

        for _, row in merged.iterrows():
            b = int(row.band)
            band_returns[b] += row.next_return
            band_counts[b] += 1

        means = np.zeros(N_BANDS)

        for i in range(N_BANDS):
            if band_counts[i] > 0:
                means[i] = band_returns[i] / band_counts[i]
            else:
                means[i] = np.nan

        means = (
            pd.Series(means)
            .interpolate(limit_direction="both")
            .fillna(0)
            .tolist()
        )

        results[sym] = {
            "band_returns": means,
            "band_counts": band_counts.tolist(),
            "n_headlines": len(sub),
            "n_days": len(merged),
            "score_mean": float(merged.sentiment.mean()),
            "score_std": float(merged.sentiment.std()),
            "daily_data": merged.to_dict("records"),
        }

    print(f"\nFinished building tables for {len(results)} stocks")

    return results

In [7]:
def _sentiment_label(score: float) -> str:
    if score > 0.6:
        return "Very Bullish 🟢🟢"
    if score > 0.2:
        return "Bullish 🟢"
    if score > -0.2:
        return "Neutral ⚪"
    if score > -0.6:
        return "Bearish 🔴"
    return "Very Bearish 🔴🔴"


def predict_next_close(
    ticker: str,
    band_tables: dict,
    model: SentimentModel,
) -> dict:

    ticker = ticker.upper()
    print(f"\nAnalyzing {ticker}…")

    # ---------------- Price ----------------
    price_data = fetch_current_price(ticker)
    if price_data.get("current_price") is None:
        return {
            "error": price_data.get("error", "Price fetch failed"),
            "ticker": ticker,
        }

    current_price = price_data["current_price"]

    # ---------------- News ----------------
    news = fetch_today_news(ticker)

    if news:
        titles = [n["title"] for n in news]
        scores = model.score_batch(titles)

        avg_score = float(np.mean(scores))

        scored_news = [
            {**n, "sentiment_score": round(s, 4)}
            for n, s in zip(news, scores)
        ]
    else:
        _warn(f"No recent news found for {ticker}; assuming neutral sentiment.")
        avg_score = 0.0
        scored_news = []

    # ---------------- Sentiment Band ----------------
    today_band = score_to_band(avg_score)

    # ---------------- Historical Lookup ----------------
    stock_table = band_tables.get(ticker)

    if stock_table is not None and stock_table.get("n_days", 0) > 0:

        band_returns = stock_table["band_returns"]
        band_counts = stock_table["band_counts"]

        band_return_pct = float(band_returns[today_band])
        band_n = int(band_counts[today_band])

        has_history = True

    else:
        # Fallback when no historical sentiment table exists
        band_return_pct = round((today_band - 9.5) / 9.5 * 1.5, 4)
        band_counts = [0] * N_BANDS
        band_returns = [0.0] * N_BANDS
        band_n = 0

        has_history = False

    # ---------------- Prediction ----------------
    predicted_price = current_price * (1 + band_return_pct / 100)

    return {
        "ticker": ticker,
        "price_data": price_data,
        "current_price": round(current_price, 2),
        "predicted_price": round(predicted_price, 2),
        "predicted_change_pct": round(band_return_pct, 4),
        "today_sentiment_score": round(avg_score, 4),
        "today_band": today_band,
        "band_count_in_history": band_n,
        "band_counts": band_counts,
        "band_returns": band_returns,
        "news": scored_news,
        "has_history": has_history,
        "sentiment_label": _sentiment_label(avg_score),
        "analysis_time": datetime.now().isoformat(),
    }

In [8]:
def display_prediction(r: dict):
    if "error" in r:
        _err(f"{r['ticker']}: {r['error']}")
        return

    sym = r["ticker"]
    cp = r["current_price"]
    pp = r["predicted_price"]
    pct = r["predicted_change_pct"]
    sc = r["today_sentiment_score"]
    band = r["today_band"]
    lbl = r["sentiment_label"]
    pd_ = r["price_data"]
    arr = "▲" if pct >= 0 else "▼"

    _hr(f"  {sym}  —  {pd_.get('name', sym)}")

    print(f"  Current price     : ${cp:.2f}")
    print(f"  Open / High / Low : ${pd_.get('open',0):.2f} / ${pd_.get('high',0):.2f} / ${pd_.get('low',0):.2f}")
    print(f"  Prev close        : ${pd_.get('prev_close',0):.2f}")
    print(f"  Day change        : {pd_.get('day_change_pct',0):+.2f}%")
    print(f"  Volume            : {pd_.get('volume',0):,}")
    print()

    print(f"  Sentiment score   : {sc:+.4f} ({lbl})")

    if r["has_history"]:
        print(f"  Sentiment band    : B{band:02d} / {N_BANDS-1} ({r['band_count_in_history']} observations)")
    else:
        print(f"  Sentiment band    : B{band:02d} / {N_BANDS-1} (No historical data)")

    print(f"  Predicted close   : ${pp:.2f}   {arr} {pct:+.2f}%")

    if r["has_history"]:

        print(f"\n  {'─'*52}")
        print("  20 Sentiment Bands — historical average next-day return")
        print(f"  {'─'*52}")

        br = r["band_returns"]
        bc = r["band_counts"]

        maxabs = max(abs(x) for x in br) or 1

        for i, ret in enumerate(br):
            active = i == band

            bar = ("█" * int(abs(ret) / maxabs * 20)).ljust(20)

            arrow = "▲" if ret >= 0 else "▼"
            sign = "+" if ret >= 0 else ""
            obs = bc[i]

            line = (
                f"B{i:02d}  {arrow} {bar} "
                f"{sign}{ret:.2f}%  (n={obs})"
            )

            if active:
                print("► " + line + " ◀ TODAY")
            else:
                print("  " + line)

    if r.get("news"):

        print(f"\n  {'─'*52}")
        print(f"  Today's News ({len(r['news'])} headlines)")
        print(f"  {'─'*52}")

        for item in r["news"][:8]:
            score = item["sentiment_score"]

            if score > 0.1:
                tag = "POS"
            elif score < -0.1:
                tag = "NEG"
            else:
                tag = "NEU"

            title = textwrap.shorten(item["title"], width=80, placeholder="…")
            print(f"  [{tag} {score:+.2f}] {title}")

    print()


def display_comparison(results: list):
    _hr("Multi-Stock Comparison")

    print(f"{'Ticker':<8}{'Price':>10}{'Pred Close':>14}{'Pred %':>10}{'Sentiment':>12}{'Band':>8}")
    print("-" * 70)

    for r in results:

        if "error" in r:
            print(f"{r['ticker']:<8} ERROR: {r['error']}")
            continue

        arrow = "▲" if r["predicted_change_pct"] >= 0 else "▼"

        print(
            f"{r['ticker']:<8}"
            f"${r['current_price']:>9.2f}"
            f"${r['predicted_price']:>13.2f}"
            f"{arrow}{r['predicted_change_pct']:>8.2f}%"
            f"{r['today_sentiment_score']:>12.3f}"
            f"   B{r['today_band']:02d}"
        )

    print()


def display_top_stocks(top_stocks: list, df: pd.DataFrame):

    counts = df["stock"].value_counts()
    max_count = counts.iloc[0]

    _hr("Top Stocks by News Coverage")

    print(f"{'#':<4}{'Ticker':<8}{'Headlines':>12}  Coverage")
    print("-" * 55)

    for i, sym in enumerate(top_stocks[:20], 1):
        count = counts.get(sym, 0)
        bar = "█" * int(count / max_count * 25)

        print(f"{i:<4}{sym:<8}{count:>12,}  {bar}")

    print()


# ---------------------------------------------------------------------
# CACHE
# ---------------------------------------------------------------------

CACHE_FILE = "band_tables_cache.json"


def save_band_tables(tables: dict, path: str = CACHE_FILE):
    with open(path, "w") as f:
        json.dump(tables, f, default=str, indent=2)

    _ok(f"Band tables saved → {path}")


def load_band_tables(path: str = CACHE_FILE):

    if not os.path.exists(path):
        return None

    try:
        with open(path) as f:
            tables = json.load(f)

        _ok(f"Loaded cached band tables ({len(tables)} stocks)")
        return tables

    except Exception as e:
        _warn(f"Could not load cache: {e}")
        return None



In [9]:
SYSTEM_PROMPT = """You are SentimentEdge.

You are a STRICT financial computation engine.

RULES:
- You DO NOT explain predictions
- You DO NOT compare stocks in natural language
- You ONLY restate provided numerical outputs
- You MUST NOT infer reasons or causes
- You MUST NOT rank stocks using reasoning words like "better", "favorable"

OUTPUT FORMAT RULES:
1. If single stock → return structured bullet points only
2. If comparison → return sorted table + highest return ticker only
3. No narrative sentences allowed

You may ONLY use:
- numbers
- tickers
- sentiment scores
- band values
- predicted returns

END MESSAGE FORMAT:
⚠ Educational analysis only — not financial advice.
"""


class StockChatbot:
    def __init__(self, groq_key: str, df: pd.DataFrame, prices: dict,
                 band_tables: dict, model: SentimentModel, top_stocks: list):

        self.client = Groq(
            api_key=groq_key,
        )

        self.df = df
        self.prices = prices
        self.band_tables = band_tables
        self.model = model
        self.top_stocks = top_stocks
        self.history: list = []

    # ── intent + ticker detection ─────────────────────────────────────────────
    def extract_tickers(self, text: str) -> list:
        valid = set(self.top_stocks) | {
            "AAPL","MSFT","GOOGL","AMZN","TSLA","NVDA","META","AMD","NFLX",
            "JPM","BAC","WMT","XOM","CVX","V","MA","DIS","INTC","ORCL","IBM",
        }

        found = []

        for w in text.upper().split():
            c = "".join(ch for ch in w if ch.isalpha())
            if 1 <= len(c) <= 5 and c in valid and c not in found:
                found.append(c)

        return found[:5]

    def detect_intent(self, text: str) -> str:
        t = text.lower()

        if any(k in t for k in ["top","list","rank","coverage","dataset"]):
            return "top_stocks"

        if any(k in t for k in ["compare","vs","versus","against"]):
            return "compare"

        if any(k in t for k in ["band","history","historical","table"]):
            return "bands"

        if any(k in t for k in ["news","headline","sentiment","score"]):
            return "sentiment"

        if any(k in t for k in ["predict","tomorrow","forecast","close","price","next"]):
            return "predict"

        if any(k in t for k in ["help","what can","how","explain"]):
            return "help"

        return "general"

    # ── data pipeline ─────────────────────────────────────────────────────────
    def gather_context(self, user_input: str) -> str:

        tickers = self.extract_tickers(user_input)
        intent = self.detect_intent(user_input)

        parts = []

        if intent == "top_stocks":

            display_top_stocks(self.top_stocks, self.df)

            counts = self.df["stock"].value_counts()

            parts.append(
                "Top-10: " +
                ", ".join(f"{s}({counts[s]:,})" for s in self.top_stocks[:10])
            )

        elif intent == "compare" and len(tickers) >= 2:

            results = [
                predict_next_close(s, self.band_tables, self.model)
                for s in tickers
            ]

            display_comparison(results)

            parts.append(
                "Comparison: " +
                json.dumps(
                    [
                        {
                            k: v
                            for k, v in r.items()
                            if k in (
                                "ticker",
                                "current_price",
                                "predicted_price",
                                "predicted_change_pct",
                                "today_sentiment_score",
                                "today_band",
                                "sentiment_label",
                            )
                        }
                        for r in results
                        if "error" not in r
                    ],
                    indent=2,
                )
            )

        elif tickers:

            for sym in tickers[:2]:

                r = predict_next_close(sym, self.band_tables, self.model)

                display_prediction(r)

                slim = {
                    k: v
                    for k, v in r.items()
                    if k not in (
                        "daily_data",
                        "band_counts",
                        "band_returns",
                        "news",
                    )
                }

                slim["top_headlines"] = [
                    {
                        "title": n["title"][:80],
                        "score": n["sentiment_score"],
                    }
                    for n in r.get("news", [])[:5]
                ]

                slim["band_summary"] = {
                    f"B{b:02d}": r.get("band_returns", [])[b]
                    for b in range(N_BANDS)
                    if r.get("band_returns")
                }

                parts.append(
                    f"Analysis for {sym}:\n" +
                    json.dumps(slim, indent=2, default=str)
                )

        elif intent == "help":

            parts.append(
                f"SentimentEdge: predict next-day close, compare stocks, "
                f"show sentiment bands, top stocks by coverage. "
                f"Dataset: 1.41M headlines 2009-2020, {len(self.top_stocks)} stocks."
            )

        return "\n\n".join(parts)

    # ── Groq call ─────────────────────────────────────────────────────────────
    def chat(self, user_input: str) -> str:

        context = self.gather_context(user_input)

        msg = user_input + (
            f"\n\n[Data]\n{context}" if context else ""
        )

        self.history.append(
            {
                "role": "user",
                "content": msg,
            }
        )

        msgs = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            }
        ] + self.history[-8:]

        try:

            resp = self.client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=msgs,
                temperature=0.25,
                max_completion_tokens=800,
            )

            reply = resp.choices[0].message.content

        except Exception as e:

            reply = f"Groq error: {e}"

        self.history.append(
            {
                "role": "assistant",
                "content": reply,
            }
        )

        return reply

    # ── terminal loop ─────────────────────────────────────────────────────────
    def run_terminal(self):

        _hr("SentimentEdge — Stock Prediction Chatbot")

        print(
            "Commands: ticker (AAPL), compare (TSLA vs NVDA), "
            "top stocks, help, quit\n"
        )

        while True:

            try:
                user = input("You → ").strip()

            except (EOFError, KeyboardInterrupt):
                print("\nGoodbye.")
                break

            if not user:
                continue

            if user.lower() in {"quit", "exit", "q", "bye"}:
                print("Goodbye.")
                break

            reply = self.chat(user)

            print(f"\n🤖 SentimentEdge:\n{reply}\n")

In [10]:
def setup(
    groq_key: str,
    top_n: int = 100,
    start_date: str = "2009-01-01",
    end_date: str = str(date.today()),
    use_finbert: bool = True,
    no_cache: bool = False,
) -> "StockChatbot":
    """
    Full initialisation pipeline.
    Call once per Colab session.
    """

    groq_key = groq_key or os.environ.get("GROQ_API_KEY", "")

    if not groq_key:
        raise ValueError("Groq API key required.")

    _hr("SentimentEdge  —  Initialising")

    # 1. Load sentiment model
    model = SentimentModel(use_finbert=use_finbert)

    # 2. Load dataset
    df = load_dataset()

    # 3. Select candidate stocks
    candidates = pick_candidates(df, n=top_n)

    # 4. Download historical prices
    print(f"\nFetching historical prices ({start_date} → {end_date})...")

    prices = fetch_historical_prices(
        candidates,
        start_date,
        end_date,
    )

    # 5. Keep only active stocks
    top_stocks = filter_active(
        candidates,
        prices,
        n=top_n,
    )

    # 6. Load cached band tables if available
    band_tables = None

    if not no_cache:
        band_tables = load_band_tables()

    # 7. Otherwise build them
    if band_tables is None:
        band_tables = build_band_tables(
            df,
            prices,
            model,
            top_stocks,
        )
        save_band_tables(band_tables)

    _hr("Setup complete — ready to chat!")

    print("Examples:")
    print("chat(bot, 'Predict AAPL tomorrow')")
    print("chat(bot, 'Compare TSLA and NVDA')")
    print("chat(bot, 'What is today's sentiment for MSFT?')")
    print("launch_widget(bot)\n")

    return StockChatbot(
        groq_key,
        df,
        prices,
        band_tables,
        model,
        top_stocks,
    )


def chat(bot: "StockChatbot", user_input: str):

    reply = bot.chat(user_input)

    print("\n" + "─" * 70)
    print("🤖 SentimentEdge\n")
    print(reply)
    print("─" * 70 + "\n")

    return reply


def launch_widget(bot: "StockChatbot"):

    try:
        import ipywidgets as widgets
        from IPython.display import display, HTML

    except ImportError:
        _err("ipywidgets not installed.")
        return

    out = widgets.Output()

    text = widgets.Text(
        placeholder="Ask about any stock...",
        layout=widgets.Layout(width="80%"),
    )

    button = widgets.Button(
        description="Send",
        button_style="primary",
    )

    def send(_):

        question = text.value.strip()

        if not question:
            return

        text.value = ""

        with out:
            print("\n" + "=" * 70)
            print("You:", question)
            print("=" * 70)

            answer = bot.chat(question)

            print("\n🤖 SentimentEdge\n")
            print(answer)

    button.on_click(send)
    text.on_submit(send)

    display(HTML("<h3>💹 SentimentEdge Stock Chatbot</h3>"))
    display(widgets.HBox([text, button]))
    display(out)

In [11]:
from datetime import date
from getpass import getpass

groq_key = getpass("Enter your Groq API Key: ").strip()

bot = setup(
    groq_key=groq_key,
    top_n=100,
    start_date="2009-01-01",
    end_date=str(date.today()),
    use_finbert=True,
    no_cache=False,
)

bot.run_terminal()

Enter your Groq API Key: ··········

────────────────────────────────────────────────────────────  SentimentEdge  —  Initialising

Loading FinBERT… (downloads ~500 MB on first run)


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

...

Price fetch:   0%|          | 0/140 [00:00<?]


✅ Price data: 114 loaded | 26 delisted/empty | 0 errors
No data: BBRY, JCP, AGN, CHK, GPS, MYL, DISH, ATVI, LNKD, MON, GMCR, JWN, RAD, AKS, FCAU, FEYE, NBG, FL, JNPR, POT, K, KORS, CREE, COH, RHT, BHI
✅ Active: 114 | Delisted/no-data: 26 → using top 100
Dropped: BBRY, JCP, AGN, CHK, GPS, MYL, DISH, ATVI, LNKD, MON, GMCR, JWN, RAD, AKS, FCAU, FEYE, NBG, FL, JNPR, POT, K, KORS, CREE, COH, RHT, BHI

Building band tables for 100 stocks (196,417 headlines)



Stocks:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Finished building tables for 97 stocks
✅ Band tables saved → band_tables_cache.json

────────────────────────────────────────────────────────────  Setup complete — ready to chat!

Examples:
chat(bot, 'Predict AAPL tomorrow')
chat(bot, 'Compare TSLA and NVDA')
chat(bot, 'What is today's sentiment for MSFT?')
launch_widget(bot)


────────────────────────────────────────────────────────────  SentimentEdge — Stock Prediction Chatbot

Commands: ticker (AAPL), compare (TSLA vs NVDA), top stocks, help, quit

You → What will be the closing price of TSLA tomorrow?

Analyzing TSLA…

────────────────────────────────────────────────────────────    TSLA  —  Tesla, Inc.

  Current price     : $319.69
  Open / High / Low : $341.00 / $342.11 / $315.74
  Prev close        : $374.01
  Day change        : -14.52%
  Volume            : 115,606,400

  Sentiment score   : -0.0064 (Neutral ⚪)
  Sentiment band    : B09 / 19 (No historical data)
  Predicted close   : $319.44   ▼ -0.08%

  ────────────────────